# Analyse des annotations Qwen3.8 Gyro-S

Ce notebook se concentre exclusivement sur les lignes d’annotations produites par Qwen3.8 Gyro-S. Il analyse leur couverture, la répartition des taxonomies et de la qualité, puis montre des exemples aléatoires avec leurs images et leurs étiquettes. Les annotations sont des prédictions automatiques : `auto_labeled` ne signifie pas qu’elles ont été validées par une personne.


In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import base64
import html
import json
import os
import random
from io import BytesIO

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import HTML, display
from huggingface_hub import HfApi, hf_hub_download

DATASET_REPO = 'ethanzxv/pixelart-dataset'
ANNOTATION_FILE_OVERRIDE = None  # Optionnel : chemin JSONL exact dans le dépôt HF.
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN') or True

# Racine locale du projet et JSONL produit par le run courant.
cwd = Path.cwd().resolve()
project_roots = [p for p in (cwd, *cwd.parents) if (p / 'notebook/dataset').is_dir()]
PROJECT_ROOT = project_roots[0] if project_roots else cwd
LOCAL_ANNOTATIONS = PROJECT_ROOT / 'notebook/dataset/taxonomy/taxonomy_work/annotations_v3_4_full_color.jsonl'

# Priorité : override explicite, fichier vivant local, puis snapshot Qwen3.8 le plus récent sur HF.
if ANNOTATION_FILE_OVERRIDE:
    annotation_path = Path(hf_hub_download(repo_id=DATASET_REPO, repo_type='dataset', filename=ANNOTATION_FILE_OVERRIDE, token=HF_TOKEN))
    annotation_source = ANNOTATION_FILE_OVERRIDE
elif LOCAL_ANNOTATIONS.is_file():
    annotation_path = LOCAL_ANNOTATIONS
    annotation_source = f'workspace · {LOCAL_ANNOTATIONS}'
else:
    api = HfApi(token=HF_TOKEN)
    files = api.list_repo_files(DATASET_REPO, repo_type='dataset')
    remote_candidates = [
        name for name in files
        if name.endswith('.jsonl') and (
            name.startswith('annotations/dataset_lab_sync/')
            or name.startswith('annotations/taxonomy_runs/')
        )
    ]
    remote_candidates.sort(reverse=True)
    annotation_path = None
    annotation_source = None
    for remote_name in remote_candidates:
        downloaded = Path(hf_hub_download(repo_id=DATASET_REPO, repo_type='dataset', filename=remote_name, token=HF_TOKEN))
        try:
            with downloaded.open(encoding='utf-8') as stream:
                first = next((json.loads(line) for line in stream if line.strip()), {})
            model_name = str(first.get('model', '')).lower()
            if 'qwen3.8' in model_name and 'gyro' in model_name:
                annotation_path, annotation_source = downloaded, remote_name
                break
        except (OSError, json.JSONDecodeError):
            continue
    if annotation_path is None:
        raise FileNotFoundError('Aucun JSONL Qwen3.8 Gyro-S trouvé localement ou dans les snapshots HF.')

# Snapshot cohérent du JSONL : une éventuelle dernière ligne incomplète est ignorée.
qwen_rows = []
with annotation_path.open('rb') as stream:
    while True:
        line = stream.readline()
        if not line or not line.endswith(b'\n'):
            break
        if line.strip():
            qwen_rows.append(json.loads(line))
qwen_rows = [
    row for row in qwen_rows
    if 'qwen3.8' in str(row.get('model', '')).lower()
    and 'gyro' in str(row.get('model', '')).lower()
]
if not qwen_rows:
    raise ValueError(f'Aucune annotation Qwen3.8 Gyro-S dans {annotation_source}')

print(f'{len(qwen_rows):,} lignes Qwen3.8 Gyro-S · {len({r.get("asset_id") for r in qwen_rows}):,} asset_id uniques')
print('Source :', annotation_source)


## État général des annotations

Les compteurs portent sur le JSONL Qwen3.8 chargé ci-dessus. Les doublons sont calculés sur `asset_id` ; les champs manquants et les statuts automatiques sont exposés séparément.


In [ ]:
asset_ids = [row.get('asset_id') for row in qwen_rows]
model_counts = Counter(row.get('model', '(absent)') for row in qwen_rows)
review_counts = Counter(row.get('review_status', '(absent)') for row in qwen_rows)
quality_counts = Counter(row.get('labels', {}).get('visual_quality', '(absent)') for row in qwen_rows)
rework_counts = Counter(row.get('need_rework', '(absent)') for row in qwen_rows)
timestamps = [row.get('annotated_at_utc') for row in qwen_rows if row.get('annotated_at_utc')]
summary = pd.DataFrame([
    ('Lignes Qwen3.8 Gyro-S', len(qwen_rows)),
    ('asset_id uniques', len(set(asset_ids))),
    ('asset_id dupliqués', len(asset_ids) - len(set(asset_ids))),
    ('Première annotation (UTC)', min(timestamps) if timestamps else 'inconnue'),
    ('Dernière annotation (UTC)', max(timestamps) if timestamps else 'inconnue'),
    ('Modèles', ', '.join(f'{k}: {v}' for k, v in model_counts.items())),
], columns=['Mesure', 'Valeur'])
display(summary)
print('Statuts de revue :', dict(review_counts))
print('Qualité visuelle :', dict(quality_counts))
print('need_rework :', dict(rework_counts))


## Répartition des taxonomies

Chaque graphique présente jusqu’aux 12 valeurs les plus fréquentes d’un champ. Les listes, comme `dominant_color_families`, comptent une image au plus une fois par valeur. `unknown` et `not_applicable` sont visibles dans les répartitions, car ils renseignent la couverture réelle du modèle.


In [ ]:
TAXONOMY_FIELDS = [
    'asset_type', 'subject_type', 'view', 'pose', 'action',
    'background_type', 'render_family', 'body_plan', 'wings',
    'background_scene', 'visual_quality', 'dominant_color_families',
    'palette_size', 'palette_temperature', 'palette_structure', 'palette_contrast',
]

def values_for(row, field):
    value = row.get('labels', {}).get(field)
    values = value if isinstance(value, list) else [value]
    return [str(value) if value is not None else '(absent)' for value in values]

fig, axes = plt.subplots(4, 4, figsize=(22, 20))
for ax, field in zip(axes.flat, TAXONOMY_FIELDS):
    counts = Counter()
    for row in qwen_rows:
        counts.update(set(values_for(row, field)))
    top = counts.most_common(12)
    if top:
        names, numbers = zip(*top)
        ax.barh(list(names)[::-1], list(numbers)[::-1], color='#5686c5')
        ax.set_xlim(0, max(numbers) * 1.18)
    ax.set_title(field, loc='left', fontsize=10, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=.2)
    ax.set_axisbelow(True)
axes.flat[-1].axis('off')
fig.suptitle('Annotations Qwen3.8 Gyro-S · répartition par champ', fontsize=15, y=1.01)
fig.tight_layout()
plt.show()


## Qualité, rework et couverture

`visual_quality` décrit l’état visuel estimé par Qwen. `need_rework` est une autre prédiction ; les deux valeurs sont comparées pour repérer les éventuelles incohérences. Les annotations `auto_labeled` restent à vérifier même si elles sont évaluées `clean`.


In [ ]:
quality_audit = Counter(
    (row.get('labels', {}).get('visual_quality', '(absent)'), row.get('need_rework', '(absent)'))
    for row in qwen_rows
)
audit_df = pd.DataFrame([
    {'visual_quality': quality, 'need_rework': rework, 'images': count}
    for (quality, rework), count in quality_audit.most_common()
])
display(audit_df)

coverage_rows = []
for field in TAXONOMY_FIELDS:
    missing = sum(not row.get('labels', {}).get(field) for row in qwen_rows)
    unknown = sum('unknown' in values_for(row, field) for row in qwen_rows)
    not_applicable = sum('not_applicable' in values_for(row, field) for row in qwen_rows)
    coverage_rows.append({
        'champ': field,
        'manquant': missing,
        'unknown': unknown,
        'not_applicable': not_applicable,
        'renseigné': len(qwen_rows) - missing,
    })
display(pd.DataFrame(coverage_rows).set_index('champ'))


## Analyse des couleurs et palettes

Ces tableaux détaillent les familles de couleurs et les caractéristiques de palette proposées par Qwen3.8. Les familles comptent le nombre d’images contenant chaque couleur. La taille, la température, la structure et le contraste sont des catégories estimées, pas des mesures colorimétriques exactes.


In [ ]:
color_counts = Counter()
for row in qwen_rows:
    color_counts.update(set(values_for(row, 'dominant_color_families')))
color_table = pd.DataFrame(color_counts.most_common(), columns=['famille de couleur', 'images'])
display(color_table)

for field in ('palette_size', 'palette_temperature', 'palette_structure', 'palette_contrast'):
    counts = Counter(row.get('labels', {}).get(field, '(absent)') for row in qwen_rows)
    display(pd.DataFrame(counts.most_common(), columns=[field, 'images']))

palette_cross = pd.crosstab(
    [row.get('labels', {}).get('palette_size', '(absent)') for row in qwen_rows],
    [row.get('labels', {}).get('palette_temperature', '(absent)') for row in qwen_rows],
    margins=True,
)
display(palette_cross)


## Exemples aléatoires d’annotations

Change `N_EXEMPLES` ou `GRAINE` pour parcourir un autre échantillon. Les images sont recherchées dans le cache local puis téléchargées depuis le dataset HF privé si nécessaire. Le HTML affiche la caption Qwen et toutes les taxonomies de chaque exemple.


In [ ]:
N_EXEMPLES = 12
GRAINE = 42
TELECHARGER_IMAGES_HF = True  # False pour n’afficher que les images déjà en cache.

cache_roots = [PROJECT_ROOT / 'notebook/training/iris_sprite_lora_work/source_images']
image_sample = random.Random(GRAINE).sample(qwen_rows, min(N_EXEMPLES, len(qwen_rows)))
cards = []
for row in image_sample:
    source = row.get('source', {})
    relative_path = str(source.get('relative_path', '')).replace('\\', '/')
    root_name = str(source.get('root', 'data'))
    image_path = next((root / root_name / relative_path for root in cache_roots if (root / root_name / relative_path).is_file()), None)
    if image_path is None and TELECHARGER_IMAGES_HF and relative_path:
        try:
            image_path = Path(hf_hub_download(
                repo_id=DATASET_REPO, repo_type='dataset',
                filename=f'data/{relative_path}', token=HF_TOKEN,
            ))
        except Exception as exc:
            image_path = None
            image_error = str(exc)[:180]
    image_html = '<p><em>Image absente du cache ou inaccessible sur HF.</em></p>'
    if image_path is not None and image_path.is_file():
        try:
            with Image.open(image_path) as image:
                image_size = f'{image.width} × {image.height} px'
                preview = image.convert('RGBA')
                preview.thumbnail((256, 256), Image.Resampling.NEAREST)
                buffer = BytesIO()
                preview.save(buffer, format='PNG')
            encoded = base64.b64encode(buffer.getvalue()).decode('ascii')
            image_html = f'<img src="data:image/png;base64,{encoded}" style="width:192px;height:192px;object-fit:contain;image-rendering:pixelated;background:#eee"><p>{image_size}</p>'
        except OSError:
            image_html = '<p><em>Fichier image illisible.</em></p>'
    labels_html = html.escape(json.dumps(row.get('labels', {}), ensure_ascii=False, indent=2))
    caption_html = html.escape(str(row.get('caption_candidate', '(caption absente)')))
    asset_id = html.escape(str(row.get('asset_id', '(id absent)')))
    source_html = html.escape(relative_path)
    cards.append(
        f'<article style="flex:1;min-width:270px;max-width:390px;padding:12px;border:1px solid #ccc;border-radius:8px">'
        f'<h4>{asset_id}</h4>{image_html}<p><b>Source :</b> {source_html}</p>'
        f'<p><b>Qualité :</b> {html.escape(str(row.get("labels", {}).get("visual_quality", "(absent)")))} · '
        f'<b>Rework :</b> {html.escape(str(row.get("need_rework", "(absent)")))}</p>'
        f'<p><b>Caption Qwen :</b> {caption_html}</p>'
        f'<details><summary>Taxonomies complètes</summary><pre style="white-space:pre-wrap;font-size:11px">{labels_html}</pre></details></article>'
    )
display(HTML('<div style="display:flex;gap:12px;flex-wrap:wrap;align-items:flex-start">' + ''.join(cards) + '</div>'))
